In [50]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from glob import glob
import torchaudio
import torchinfo
import torch
import math

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
mouseFLAC  = glob("vctk/gen/flac/wav48_silence_trimmed/*/*.flac")
stockFLAC = [m_fl.replace("vctk/gen/flac", "vctk/stock") for m_fl in mouseFLAC]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: {}".format(device))
torch.set_default_device(device)

device: cuda


In [34]:
Fs=16000
def loadFLAC(flac_fn):
    flacData, sr = torchaudio.load(flac_fn)
    # resample to Fs
    flacData = torchaudio.transforms.Resample(sr, Fs)(flacData)
    return flacData[0][1:].to(device)

#flacData, sr = torchaudio.load(mouseFLAC[0])
#plt.plot(flacData[0])

In [55]:
f=loadFLAC(mouseFLAC[0])
#f=f.reshape(1,1,len(f))
print(f.shape)
print(f.get_device())

q=2**math.ceil(math.log2(q))
print(q)


###
###
### TODO: CONVERT ALL INPUTS TO POWER-OF-TWO LENGTH
###
###

torch.Size([76191])
0
131072


In [57]:
from audio_diffusion_pytorch import DiffusionModel, UNetV0, VDiffusion, VSampler

model = DiffusionModel(
    net_t=UNetV0, # The model type used for diffusion (U-Net V0 in this case)
    in_channels=2, # U-Net: number of input/output (audio) channels
    channels=[8, 32, 64, 128, 256, 512, 512, 1024, 1024], # U-Net: channels at each layer
    factors=[1, 4, 4, 4, 2, 2, 2, 2, 2], # U-Net: downsampling and upsampling factors at each layer
    items=[1, 2, 2, 2, 2, 2, 2, 4, 4], # U-Net: number of repeating items at each layer
    attentions=[0, 0, 0, 0, 0, 1, 1, 1, 1], # U-Net: attention enabled/disabled at each layer
    attention_heads=8, # U-Net: number of attention heads per attention item
    attention_features=64, # U-Net: number of attention features per attention item
    diffusion_t=VDiffusion, # The diffusion method used
    sampler_t=VSampler, # The diffusion sampler used
)


# Train model with audio waveforms
audio = torch.randn(1, 2, q-1) # [batch_size, in_channels, length]
#audio = f.reshape(1,1,len(f))

#torchinfo.summary(model, f.shape)


print(audio.shape)
#print(len(audio[0][0]))

loss = model(audio)
loss.backward()

# Turn noise into new audio sample with diffusion
noise = torch.randn(1, 2, q-1) 
#noise = torch.randn(1, 1, len(audio[0][0])) # [batch_size, in_channels, length]
sample = model.sample(noise, num_steps=10) # Suggested num_steps 10-100


torch.Size([1, 2, 131071])


RuntimeError: The size of tensor a (127) must match the size of tensor b (126) at non-singleton dimension 2

In [17]:
print(sample.shape)

torch.Size([1, 1, 262144])
